In [87]:
import pandas as pd

#load dataset
df = pd.read_csv('C:/Users/Hubert PC/Downloads/mining_rul_system/equipment_failure.csv')


In [88]:
def clean_column_names(df):
    df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
    return df
df = clean_column_names(df)

In [89]:
def basic_cleaning(df):
    df["event_date"] = pd.to_datetime(
                                    df["event_date"],
                                    format="%m/%d/%Y",
                                    errors="coerce")
    df["operating_hours"] = pd.to_numeric(df["operating_hours"], errors="coerce")
    df["downtime_hours"] = pd.to_numeric(df["downtime_hours"], errors="coerce")
    df["repair_cost_usd"] = pd.to_numeric(df["repair_cost_usd"], errors="coerce")
    df["production_loss_tonnes"] = pd.to_numeric(df["production_loss_tonnes"], errors="coerce")
    

    return df
df = basic_cleaning(df)

In [90]:
df.dtypes 

record_id                          object
equipment_id                       object
equipment_type                     object
manufacturer                       object
model_year                          int64
mine_type                          object
country                            object
operating_hours                     int64
event_date                 datetime64[ns]
event_type                         object
failure_mode                       object
component_failed                   object
severity                           object
downtime_hours                    float64
repair_cost_usd                   float64
production_loss_tonnes              int64
maintenance_type                   object
was_scheduled                        bool
days_since_last_service             int64
oil_analysis_flag                    bool
vibration_flag                       bool
temperature_flag                     bool
dtype: object

In [91]:
df["operating_hours"].head(10)

0    64584
1    19253
2    52894
3    18369
4    27901
5    51965
6    36480
7    62968
8    18033
9    48171
Name: operating_hours, dtype: int64

check if an id is used incorrectly(duplicate use in multiple equipment type, different manufacturers using same id, one id used for two equipment bought in a different year)

In [92]:
bad_ids = df.groupby("equipment_id")[[
    "equipment_type", "manufacturer", "model_year"
]].nunique()

bad_ids = bad_ids[
    (bad_ids["equipment_type"] > 1) |
    (bad_ids["manufacturer"] > 1) |
    (bad_ids["model_year"] > 1)
]

print(len(bad_ids))
print(df["equipment_id"].nunique())

957
3825


check for any valid ids and see if they appear in more than 1 event
make multi_event_df

In [93]:
# machine_counts = df.groupby([
#     "equipment_id",
#     "equipment_type",
#     "manufacturer",
#     "model_year"
# ]).size().reset_index(name="event_count")

In [94]:
# print(machine_counts)

In [95]:
# multi_event_machines = machine_counts[machine_counts["event_count"] > 1]
# multi_event_machines_ids = multi_event_machines["equipment_id"]
# multi_event_df = df[df["equipment_id"].isin(multi_event_machines_ids)]
# single_event_df = df[~df["equipment_id"].isin(multi_event_machines_ids)]

In [96]:
# print(multi_event_machines)

In [97]:
# print(multi_event_df)

check for any valid ids and see if they appear in more than 1 event
make multi_event_df

In [98]:
keys = ["equipment_id", "equipment_type", "manufacturer", "model_year"]

multi_event_keys = df.groupby(keys).size().reset_index(name="n").query("n > 1")[keys]

multi_event_df = df.merge(multi_event_keys, on=keys)
single_event_df = df.merge(multi_event_keys, on=keys, how="left", indicator=True)\
                    .query('_merge == "left_only"')\
                    .drop(columns="_merge")

In [99]:
print(len(multi_event_df)+len(single_event_df))

5000


In [100]:
print(single_event_df.head(5))

      record_id equipment_id equipment_type manufacturer  model_year  \
0  EQP-ADB3590D    UNIT-1803     haul_truck  Caterpillar        2008   
1  EQP-596EE017    UNIT-7964        crusher      Sandvik        2019   
2  EQP-23341C24    UNIT-6890           pump        Metso        2011   
3  EQP-D8EF5309    UNIT-4949         loader        Volvo        2019   
4  EQP-5A611650    UNIT-4896      excavator      Hitachi        2017   

          mine_type       country  operating_hours event_date  \
0       underground           drc            64584 2022-12-03   
1          open_pit      botswana            19253 2023-08-03   
2          open_pit  south_africa            52894 2020-05-28   
3  processing_plant        zambia            18369 2024-11-21   
4       underground         ghana            27901 2022-12-30   

        event_type  ... severity downtime_hours repair_cost_usd  \
0        breakdown  ...    major           17.4        70862.60   
1        breakdown  ...    major          

drop identical rows

In [101]:
single_event_df = single_event_df.drop_duplicates()

Random Id Generator

In [102]:
keys = ["equipment_id", "equipment_type", "manufacturer", "model_year"]
existing_ids = set(multi_event_df["equipment_id"].unique())

import random

def generate_id(used_ids):
    while True:
        new_id = f"UNIT-{random.randint(1000, 9999)}"
        if new_id not in used_ids:
            used_ids.add(new_id)
            return new_id
        
print(existing_ids)

{'UNIT-3881', 'UNIT-7877', 'UNIT-1077'}


In [103]:
used_ids = set(existing_ids)

#create mapping: (type, manufacturer, year) -> new_id
group_to_id = {}

new_ids = []

for _, row in single_event_df.iterrows():
    group_key = tuple(row[k] for k in keys)

    if group_key not in group_to_id:
        group_to_id[group_key] = generate_id(used_ids)
    
    new_ids.append(group_to_id[group_key])

single_event_df = single_event_df.copy()
single_event_df["equipment_id"] = new_ids

In [104]:
# Same group = same ID
single_event_df.groupby(keys)["equipment_id"].nunique().value_counts()

equipment_id
1    4994
Name: count, dtype: int64

In [105]:
# No collision with multi-event IDs
set(single_event_df["equipment_id"]).intersection(existing_ids)

set()

In [67]:
single_event_df.groupby(keys).ngroups == single_event_df["equipment_id"].nunique()

True

In [107]:
df = pd.concat([multi_event_df, single_event_df], ignore_index=True)
print(df.head(10))

      record_id equipment_id equipment_type manufacturer  model_year  \
0  EQP-86409AB3    UNIT-7877         grader   John Deere        2022   
1  EQP-044D399A    UNIT-1077       conveyor     FLSmidth        2013   
2  EQP-35127C35    UNIT-7877         grader   John Deere        2022   
3  EQP-BDDE00D5    UNIT-3881     haul_truck  Caterpillar        2011   
4  EQP-3DE993DB    UNIT-3881     haul_truck  Caterpillar        2011   
5  EQP-A41E8B05    UNIT-1077       conveyor     FLSmidth        2013   
6  EQP-ADB3590D    UNIT-3813     haul_truck  Caterpillar        2008   
7  EQP-596EE017    UNIT-3943        crusher      Sandvik        2019   
8  EQP-23341C24    UNIT-6933           pump        Metso        2011   
9  EQP-D8EF5309    UNIT-1269         loader        Volvo        2019   

          mine_type       country  operating_hours event_date  \
0          open_pit           drc             8179 2020-07-22   
1          open_pit  south_africa            42231 2022-10-16   
2       unde

In [ ]:
def build_machine_table(df):
    machine = df[[
        "equipment_id",
        "equipment_type",
        "manufacturer",
        "model_year"
    ]].drop_duplicates()

    machine = machine.rename(columns={
        "equipment_id": "machine_id",
        "model_year": "year"
    })

    return machine.reset_index(drop=True)

machine_table = build_machine_table(df)

In [ ]:
machine_table.head(10)

In [ ]:
def build_time_series_table(df):
    ts = df[[
        "equipment_id",
        "event_date",
        "operating_hours"
    ]].copy()

    ts = ts.rename(columns={
        "equipment_id" : "machine_id"
        "event_data" : "timestamp" 
    })

    # optional columns(add if exist later)
    ts["engine_hours"] = None
    ts["fuel_rate"] = None
    ts["location"] = None

time_series_table = build_time_series_table(df)

In [ ]:
def build_sensor_table(df):
    sensor_cols = [
        "oil_analysis_flag",
        "vibration_analysisi_flag",
        "temperature_flag"
    ]

    sensor = df.melt(
        id_vars = ["equipment_id", "event_date"],
        value_vars = sensor_cols,
        var_name = "sensor_name",
        value_name = "sensor_value"
    )

    sensor = sensor.rename(columns={
        "equipment_id" : "machine_id",
        "event_date" : "timestamp"
    })

    return sensor.reset_index(drop=True)

sensor_table = build_sensor_table(df)

In [ ]:
def build_fault_table(df):
    faults = df[df["event_type"].str.lower() == "fault"][[
        "equipment_id",
        "event_date",
        "failure_mode",
        "severity",
        "component_failed"
    ]].copy()

    faults - faults.rename(columns={
        "equipment_id" : "machine_id",
        "event_date" : "timestamp",
        "failure_mode" : "fault_code",
        "component_failed" : "component"
    })

    return faults.reset_index(drop=True)

fault_table = build_fault_table(df)

In [ ]:
def build_maintenance_table(df):
    maintenance = df[df["event_type"].str,lower() == "maintenance"][[
        "equipment_id",
        "event_date",
        "maintenance_type",
        "component_failed",
        "repair_cost_usd",
        "downtime_hours"
    ]].copy()

    maintenance = maintenance.rename(columns={
        "equipment_id" : "machine_id",
        "event_date" : "timestamp",
        "component_failed" : "component",
        "repair_cost_usd" : "cost"
    })

    return maintenance.reset_index(drop=True)

maintenance_table = build_maintenance_table(df)

In [ ]:
def build_failure_table(df):
    failure = df[df["event_type"].str.lower() == "failure"][[
        "equipment_id",
        "event_date",
        "failure_mode"
    ]].copy()

    failure = failure.rename(columns={
        "equipment_id" : "machine_id",
        "event_date" : "failure_timestamp",
        "failure_mode" : "failure_code"
    })

    return failure.reset_index(drop=True)

failure_label_table = build_failure_table(df)

In [ ]:
def build_all_tables(df):
    return {
        "machine": build_machine_table(df),
        "time_series": build_time_series_table(df),
        "sensor": build_sensor_table(df),
        "faults": build_fault_table(df),
        "maintenance": build_maintenance_table(df),
        "failure_labels": build_failure_table(df)
    }

tables = build_all_tables(df)

In [ ]:
for name, table in tables.items():
    print(f"\n{name.upper()} TABLE:")
    print(table.head())
    print(table.info())